In [3]:
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# === CONFIG ===
INPUT_FILE = "Gadwal Municipality New.xlsx"  # Replace with full path if needed
OUTPUT_FILE = "BOQ_Classwise_Gadwal_Corrected.xlsx"

# === HELPER FUNCTIONS ===
def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_class(text):
    text = str(text).lower()
    if 'k-9' in text or 'k9' in text:
        return 'DI-K9'
    elif 'k-7' in text:
        return 'DI-K7'
    elif 'm.s' in text or 'ms' in text:
        return 'MS'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'cpvc' in text:
        return 'CPVC'
    elif 'pvc' in text:
        return 'PVC'
    return None

def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

# === LOAD DATA AND MANUALLY RENAME COLUMNS ===
df = pd.read_excel(INPUT_FILE, header=2)  # 3rd row is actual header

# Rename relevant columns for consistency
df = df.rename(columns={
    df.columns[1]: "Item Description",   # Description of Item
    df.columns[2]: "Units",              # UOM
    df.columns[3]: "Estimate Rate",      # Rate
    df.columns[4]: "Quantity"            # BOQ Qty
})

# === CLEAN & TRANSFORM ===
df["class"] = df["Item Description"].apply(extract_class)
df["DIA"] = df["Item Description"].apply(extract_dia)
df["Units"] = df["Units"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')
df["Estimate Rate"] = pd.to_numeric(df["Estimate Rate"], errors='coerce')

# === FILTER PIPE MATERIAL ITEMS ===
df_pipe = df[df["Item Description"].str.contains("pipe|di|k-9|k-7|m.s|hdpe|pvc|cpvc", case=False, na=False)]

# Optional: Uncomment this if you only want "mtr" unit items
# df_pipe = df_pipe[df_pipe["Units"] == "mtr"]

# === FORMAT RESULT TABLE ===
results = []
for _, row in df_pipe.iterrows():
    results.append({
        "BOQ_NO": "Single",
        "Item Description": clean_illegal_chars(row["Item Description"]),
        "DI-K9": "Yes" if row["class"] == "DI-K9" else "",
        "DI-K7": "Yes" if row["class"] == "DI-K7" else "",
        "MS": "Yes" if row["class"] == "MS" else "",
        "HDPE": "Yes" if row["class"] == "HDPE" else "",
        "CPVC": "Yes" if row["class"] == "CPVC" else "",
        "PVC": "Yes" if row["class"] == "PVC" else "",
        "DIA": row["DIA"],
        "Estimate Rate": row["Estimate Rate"],
        "Units": row["Units"],
        "Quantity": row["Quantity"]
    })

# === EXPORT TO EXCEL ===
df_summary = pd.DataFrame(results)
df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
df_summary.to_excel(OUTPUT_FILE, index=False)

print(f"✅ Filtered BOQ data saved to: {OUTPUT_FILE}")


✅ Filtered BOQ data saved to: BOQ_Classwise_Gadwal_Corrected.xlsx


In [4]:
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# === CONFIG ===
INPUT_FILE = "Gadwal Municipality New.xlsx"       # Update with your file path
OUTPUT_FILE = "BOQ_Classwise_MTR_Only.xlsx"       # Output file

# === HELPER FUNCTIONS ===
def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_class(text):
    text = str(text).lower()
    if 'k-9' in text or 'k9' in text:
        return 'DI-K9'
    elif 'k-7' in text:
        return 'DI-K7'
    elif 'm.s' in text or 'ms' in text:
        return 'MS'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'cpvc' in text:
        return 'CPVC'
    elif 'pvc' in text:
        return 'PVC'
    return None

def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

# === LOAD AND RENAME COLUMNS ===
df = pd.read_excel(INPUT_FILE, header=2)

# Manually rename based on position
df = df.rename(columns={
    df.columns[1]: "Item Description",
    df.columns[2]: "Units",
    df.columns[3]: "Estimate Rate",
    df.columns[4]: "Quantity"
})

# === CLEAN & EXTRACT INFO ===
df["Item Description"] = df["Item Description"].astype(str)
df["class"] = df["Item Description"].apply(extract_class)
df["DIA"] = df["Item Description"].apply(extract_dia)
df["Estimate Rate"] = pd.to_numeric(df["Estimate Rate"], errors='coerce')
df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')

# Normalize units
df["Units"] = df["Units"].astype(str).str.lower().str.replace('.', '', regex=False).str.replace(' ', '')

unit_map = {
    "rmt": "mtr",
    "rm": "mtr",
    "meter": "mtr",
    "mtr": "mtr"
}
df["Units"] = df["Units"].map(unit_map).fillna(df["Units"])

# === FILTER ONLY 'mtr' PIPE ITEMS ===
df_pipe_mtr = df[
    df["Item Description"].str.contains("pipe|di|k-9|k9|k-7|m.s|ms|hdpe|pvc|cpvc", case=False, na=False) &
    (df["Units"] == "mtr")
].copy()

# === FORMAT OUTPUT ===
results = []
for _, row in df_pipe_mtr.iterrows():
    results.append({
        "BOQ_NO": "Single",
        "Item Description": clean_illegal_chars(row["Item Description"]),
        "DI-K9": "Yes" if row["class"] == "DI-K9" else "",
        "DI-K7": "Yes" if row["class"] == "DI-K7" else "",
        "MS": "Yes" if row["class"] == "MS" else "",
        "HDPE": "Yes" if row["class"] == "HDPE" else "",
        "CPVC": "Yes" if row["class"] == "CPVC" else "",
        "PVC": "Yes" if row["class"] == "PVC" else "",
        "DIA": row["DIA"],
        "Estimate Rate": row["Estimate Rate"],
        "Units": row["Units"],
        "Quantity": row["Quantity"]
    })

df_summary = pd.DataFrame(results)
df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
df_summary.to_excel(OUTPUT_FILE, index=False)

print(f"✅ Filtered items saved to: {OUTPUT_FILE}")


✅ Filtered items saved to: BOQ_Classwise_MTR_Only.xlsx
